Ensure you have latest version of boto and have bedrock model access for Nova models

In [ ]:
!pip install botocore --upgrade
!pip install boto3 --upgrade

In [ ]:
!pip show boto3

In [ ]:
import boto3 
import sagemaker

sess = sagemaker.Session()
bucket = sess.default_bucket() # Set a default S3 bucket
prefix = 'nova_video'

In [ ]:
import logging
from botocore.exceptions import ClientError

# Configure logging
logger = logging.getLogger(__name__)
logging.basicConfig(level=logging.INFO)

# AWS Configuration
AWS_ACCOUNT = boto3.client('sts').get_caller_identity().get('Account')
AWS_REGION = "us-east-1"
INFERENCE_PROFILE = f"arn:aws:bedrock:{AWS_REGION}:{AWS_ACCOUNT}:inference-profile/us.amazon.nova-pro-v1:0"

def generate_conversation(bedrock_client, model_id, input_text, s3_uri, account_id):
    """
    Sends a video analysis request to the model.
    
    Args:
        bedrock_client: The Boto3 Bedrock runtime client
        model_id (str): The model ID to use
        input_text (str): The analysis prompt
        s3_uri (str): The S3 URI of the video
        account_id (str): AWS account ID
    
    Returns:
        response (JSON): The model's analysis of the video
    """
    logger.info("Generating video analysis with model %s", model_id)
    print("account_id : ", account_id)
    print("s3_uri : ", s3_uri)
    
    # Construct the message
    message = {
        "role": "user",
        "content": [
            {
                "text": input_text
            },
            {
                "video": {
                    "format": "mp4",
                    "source": {
                        "s3Location": {
                            "uri": s3_uri,
                            "bucketOwner": account_id
                        }
                    }
                }
            }
        ]
    }

    messages = [message]

    # System message
    system = [
        {
            "text": "You are an expert traffic safety analyst."
        }
    ]

    print("INFERENCE_PROFILE : ",INFERENCE_PROFILE)
    
    # Send the request
    response = bedrock_client.converse(
        modelId=INFERENCE_PROFILE,
        messages=messages,
        system=system,
        inferenceConfig={
            "maxTokens": 300,
            "temperature": 0.3,
            "topP": 0.1,
            "stopSequences": []
        }
    )

    return response

def main():
    """
    Entrypoint for the video analysis example.
    """
    logging.basicConfig(level=logging.INFO,
                       format="%(levelname)s: %(message)s")

    # Configuration
    model_id = "amazon.nova-pro-v1:0"
    #s3url = "s3://sagemaker-us-east-1-716256856266/nova_video/TestVideo.mp4"
    s3url_front = "s3://fleet-safety-inbox-bucket/front.mp4"
    s3url_driver = "s3://fleet-safety-inbox-bucket/driver.mp4"
    s3url_side = "s3://fleet-safety-inbox-bucket/side.mp4"
    
    # Get AWS account ID
    sts_client = boto3.client('sts')
    account_id = sts_client.get_caller_identity()['Account']
    
    input_text_front = """You are an expert at analyzing multi-camera surveillance footage.
                           Vehicle front facing camera video is provided.
                           Provide detailed observations about activities, risks, and notable events. Start your analysis with serverity: high, medium, low.
                           limit 300 words."""
    input_text_driver = """You are an expert at analyzing multi-camera surveillance footage.
                           Vehicle driver facing camera video is provided.
                           Provide detailed observations about activities, risks, and notable events. Start your analysis with serverity: high, medium, low.
                           limit 300 words."""
    input_text_side = """You are an expert at analyzing multi-camera surveillance footage.
                           Vehicle side facing camera video is provided.
                           Provide detailed observations about activities, risks, and notable events. Start your analysis with serverity: high, medium, low.
                           limit 300 words."""

    try:
        # Initialize Bedrock client
        bedrock_client = boto3.client(service_name="bedrock-runtime")

        # Generate the analysis
        response_front = generate_conversation(
            bedrock_client, model_id, input_text_front, s3url_front, account_id)
        response_driver = generate_conversation(
            bedrock_client, model_id, input_text_front, s3url_driver, account_id)
        response_side = generate_conversation(
            bedrock_client, model_id, input_text_front, s3url_side, account_id)

        # Process and print the response
        output_message_front = response_front['output']['message']
        print(f"\nRole: {output_message_front['role']}")

        #for content in output_message_front['content']:
        #    print(f"\nAnalysis: {content['text']}")


        all_observations = []

        # Extract content from response
        if 'output' in response_front and 'message' in response_front['output'] and 'content' in response_front['output']['message']:
            observation_front = response_front['output']['message']['content'][0]['text']
            all_observations.append(observation_front)
            print('\n Analysis### :: \n', observation_front)
        else:
            print(f"Unexpected response format: {response_front}")

        if 'output' in response_driver and 'message' in response_driver['output'] and 'content' in response_driver['output']['message']:
            observation_driver = response_driver['output']['message']['content'][0]['text']
            all_observations.append(observation_driver)
            print('\n Analysis### :: \n', observation_driver)
        else:
            print(f"Unexpected response format: {response_driver}")
            
        if 'output' in response_side and 'message' in response_side['output'] and 'content' in response_side['output']['message']:
            observation_side = response_side['output']['message']['content'][0]['text']
            all_observations.append(observation_side)
            print('\n Analysis### :: \n', observation_side)
        else:
            print(f"Unexpected response format: {response_side}")

        
        # Print usage statistics
        token_usage = response_front['usage']
        print(f"\nFront Video Token Usage:")
        print(f"Input tokens:  {token_usage['inputTokens']}")
        print(f"Output tokens: {token_usage['outputTokens']}")
        print(f"Total tokens: {token_usage['totalTokens']}")
        print(f"Stop reason: {response_front['stopReason']}")

        token_usage = response_driver['usage']
        print(f"\nDriver VideoToken Usage:")
        print(f"Input tokens:  {token_usage['inputTokens']}")
        print(f"Output tokens: {token_usage['outputTokens']}")
        print(f"Total tokens: {token_usage['totalTokens']}")
        print(f"Stop reason: {response_front['stopReason']}")

        token_usage = response_side['usage']
        print(f"\nSide Video Token Usage:")
        print(f"Input tokens:  {token_usage['inputTokens']}")
        print(f"Output tokens: {token_usage['outputTokens']}")
        print(f"Total tokens: {token_usage['totalTokens']}")
        print(f"Stop reason: {response_front['stopReason']}")


    
    except ClientError as err:
        message = err.response['Error']['Message']
        logger.error("A client error occurred: %s", message)
        print(f"A client error occurred: {message}")
    except Exception as e:
        logger.error("An error occurred: %s", str(e))
        print(f"An error occurred: {str(e)}")
    else:
        print(f"\nFinished generating analysis with model {model_id}.")

    
    # Generate final summary using Converse API
    try:
        # Combine all observations into a single text
        combined_observations = "\n\n".join(all_observations)
        print("combined_observations : ",combined_observations)
        final_response = bedrock_client.converse(
            modelId=INFERENCE_PROFILE,
            messages=[{
                "role": "user",
                "content": [{
                    "text": f"Please provide a concise summary of the following observations:\n\n{combined_observations}"
                }]
            }],
            system=[{
                "text": """You are an expert at analyzing multi-camera surveillance footage. You are provided with the video analysis summaries from Vehicle front facing camera video, driver facing camera video and Vehicle side facing camera video in that order.
                These vidoes are taken from the same vehicle, at the same time. Create a final summary taking all these summaries into consideration. 
                Assign the rating considering all these risks into consideration and lean toward assigning higher severety.
                Provide a JSON object with two fields, DO NOT provide any preamble; first one is a severity based on a risk of a high impact accident. Use high, medium, low for severity. 
                For second field, include a short paragraph summary of the event."""
            }]
        )

        # Extract content from final response
        if 'output' in final_response and 'message' in final_response['output'] and 'content' in final_response['output']['message']:
            print("\nFinal Summary###\n", final_response['output']['message']['content'][0]['text'])
        else:
            print(f"Unexpected final response format: {final_response}")


        # Print usage statistics
        token_usage = final_response['usage']
        print(f"\nFinal Summary Token Usage:")
        print(f"Input tokens:  {token_usage['inputTokens']}")
        print(f"Output tokens: {token_usage['outputTokens']}")
        print(f"Total tokens: {token_usage['totalTokens']}")
        print(f"Stop reason: {response_front['stopReason']}")

    
    except Exception as e:
        print(f"Error generating final summary: {str(e)}")
        return "Error generating summary"

if __name__ == "__main__":
    main()